Step 1: Bootstrap & install instructions

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython
from google.colab import userdata, drive
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")
print('Ready!')

Step 2: Install & verify langgraph

In [ ]:
!pip install --quiet langgraph

import importlib.metadata
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Check version
try:
    print(f"✓ langgraph {importlib.metadata.version('langgraph')}")
except importlib.metadata.PackageNotFoundError:
    raise RuntimeError("langgraph not installed — run !pip install langgraph")

# 2. Build and run a minimal graph to verify imports and execution
class _State(TypedDict):
    status: str

def _node(state: _State) -> _State:
    return {"status": "ok"}

_g = StateGraph(_State)
_g.add_node("test", _node)
_g.add_edge(START, "test")
_g.add_edge("test", END)

result = _g.compile().invoke({"status": "start"})

if result.get("status") != "ok":
    raise AssertionError(f"Expected status 'ok', got {result}")

print("✓ Graph compiled and executed successfully!")

Step 3: Verify parity with week 2

In [ ]:
from astra_swarm.graph import graph_triage
from astra_swarm.pipeline import astra_swarm_triage  # Week 2 baseline
from astra_swarm.cassette import cassette
import json
from pathlib import Path

alerts = json.loads(
    Path("/content/astra-swarm/data/synthetic/02_alerts.json").read_text()
)

with cassette("week3_port_parity_check"):
    for a in alerts[:]:
        g = graph_triage(a)
        # Same result shape, different container
        print(f"Route: {g['routing'].alert_class.value}, "
              f"Sev: {g['investigation'].severity.value}, "
              f"Rounds: {g['investigation'].rounds_used}")

Step 4: Visualize the graph

In [ ]:
from astra_swarm.graph import triage_graph
from IPython.display import Image, display
display(Image(triage_graph.get_graph().draw_mermaid_png()))

Step 5

In [ ]:
with cassette("week3_evaluator_test"):
    for a in alerts[:]:
        result = graph_triage(a)
        e = result["evaluation"]
        r = result.get("refinement_count", 0)
        print(f"pass={e.overall_pass}  refinements={r}  "
              f"scores=({e.completeness:.2f}, {e.citation_quality:.2f}, "
              f"{e.severity_defensibility:.2f})")